RAG Pipeline - Data Ingestion to Vector DB

In [9]:
import os

from langchain_classic import text_splitter
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
#Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    "Process all the pdf files in a directory"
    all_documents = []
    pdf_dir = Path(pdf_directory)


    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source-_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")


        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process
Processing ../data/pdf/ml_concepts.pdf
 Loaded 1 pages
Processing ../data/pdf/sample_resume.pdf
 Loaded 1 pages
Processing ../data/pdf/ai_basics.pdf
 Loaded 1 pages
Processing ../data/pdf/story.pdf
 Loaded 1 pages

Total documents loaded: 4


In [14]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/sample_resume.pdf', 'total_pages': 1, 'page'

In [32]:
def split_documents(documents,chunk_size=500,chunk_overlap=50):
    "Split a list of documents into chunks of size chunk_size"
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ',' ']
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:20]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [33]:
chunks = split_documents(all_pdf_documents)
chunks

Split 4 documents into 4 chunks

Example chunk:
Content: Machine Learning Con...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/ml_concepts.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source-_file': 'ml_concepts.pdf', 'file_type': 'pdf'}, page_content='Machine Learning Concepts\nSupervised Learning: Linear Regression, Logistic Regression\nUnsupervised Learning: K-Means, PCA\nEvaluation Metrics: Accuracy, Precision, Recall'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-20T06:28:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-20T06:28:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/sample_resume.pdf', 'total_pages': 1, 'page'

Embedding and vector store db
